# Tutorial 2: Stochastically Solving a Spatially Heterogeneous Genetic Information Processing System #

## Introduction ##
In this tutorial, we will extend the spatial stochastic techniques we have applied to the bimolecular reaction system to a genetic information processing system inside the minimal cell. This tutorial will help you become familiar with adding additional site types such as the DNA and ribosome regions and will allow you to simulate the amount of RNA and protein species produced within a single cell cycle of the minimal cell (~6300s).

Because using spatial simulation techniques is more computationally intensive than spatially homogeneous techniques, we also provide a 6300s results file called `./Results/TutR2.GIP_result_6300s.lm`. Running this tutorial's simulation for 6300s of biological time should only take ~5h, but we have provided this results file for convienience. The tutorial is set up to run the simulation for 60s just to get a feel for using these techniques.

## Set the Working Directory and Import Packages ##
To solve this system, we will import functionalities from numpy, scipy and matplotlib as well as the general operating system package (os) to enable us to access various standard file commands. But first, we will use a predefined function to set our working directory.

In [ ]:
# Import necessary packages #
import os, sys

# Define function to find desired working directory #
def _find_tutorial_dir(notebook_filename):
    """Locate this notebook's directory by searching upward from CWD."""
    start = os.path.abspath(os.getcwd())
    search = start
    for _ in range(8):
        if os.path.isfile(os.path.join(search, notebook_filename)):
            return search
        for subdir in ['LM/RDME/Tutorial02_GeneticInformationProcessing',
                       'RDME/Tutorial02_GeneticInformationProcessing',
                       'Tutorial02_GeneticInformationProcessing']:
            candidate = os.path.abspath(os.path.join(search, subdir))
            if os.path.isfile(os.path.join(candidate, notebook_filename)):
                return candidate
        search = os.path.dirname(search)
    return None

# Define desired working directory #
_here = _find_tutorial_dir('Tut2_GeneticInformationProcessing.ipynb')
if _here is None:
    raise RuntimeError(
        "Cannot locate Tutorial02_GeneticInformationProcessing/. "
        "Please ensure the repository structure is intact."
    )

# Set working directory #
os.chdir(_here)
print(f"Working directory set to:\n{_here}")

Next, we will import other necessary packages and a custom script for loading in precomputed masks for the ribosome and DNA regions.

In [ ]:
# Import standard python libraries #
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add custom script for loading in precomputed region masks #
sys.path.append('Utils')
import T2_RibosomeAndDNAMasks as loader

# Import jLM packages and modules #
import jLM                                  # Set up the Jupyter environment
from jLM.RegionBuilder import RegionBuilder # Deal with the spatial geometry
from jLM.RDME import Sim as RDMESim         # Main simulation class
import lm                                   # Set up LM environment
from lm import IntMpdRdmeSolver             # lm::rdme::IntMpdRdmeSolver
from jLM.RDME import File as RDMEFile       # Functionality for working with RDME output files
from jLM import Lattice as LMLattice        # Lattice functionality for region-specific trajectory results
import warnings                             # jLM warnings utility function

# Define the RDME Simulation Object #
As we did in the previous tutorial, we will define all system and simulation parameters and then create an RDME simulation object. In this system, we will set the voxel edge length to 8nm and the number of voxels in each dimension to 64.

In [ ]:
# Define simulation and system parameters #
# Define simulation name #
simName = "RDME_GIP"
# Define the output filename to save results #
filename = './Results/TutR2.GIP_result_60s.lm'
# Define length of voxel edges (m) #
lattice_spacing = 8e-9
# Define number of voxels per dimension of the system #
N_edges = [64, 64, 64]
# Define default region type name #
regionN = "extracellular"
# Define lattice type as "Int" #   
lattice_type = "Int"
# Define total biological time of simulation (s) #
totalTime = 60
# Define time interval for timestep (s) #
timeStep = 50e-6            
# Define number of simulation steps before writing data: 20000 * 50e-6 = 1s #
writeInterval = 20000    


# Define the center voxel of the simulation #
sim_center = [int(N_edges[0]/2),int(N_edges[1]/2),int(N_edges[2]/2)]
# Define other lattice locations for future mask building #
N_2_x=int(N_edges[0]/2)
N_2_y=int(N_edges[1]/2)
N_2_z=int(N_edges[2]/2)

# Create RDME simulation object #
sim = RDMESim(name = simName,
              filename = filename,
              dimensions = N_edges,
              latticeSpacing = lattice_spacing,
              regionName = regionN,
              latticeType = lattice_type,
              dt = timeStep)

# Add total simulation time to RDME simulation object #
sim.simulationTime = totalTime
# Add lattice write interval to simulation object #
sim.latticeWriteInterval = writeInterval
# Add species write interval to simulation object #
sim.speciesWriteInterval = writeInterval

# Create the System Architecture #
Now that we have our simulation object created, we will construct the system geometry/architecture. For the system in this tutorial, we will have six region types:
+ `extracellular`,
+ `membrane`
+ `shell`
+ `cytoplasm`
+ `ribosomes`, and
+ `DNA`

First, we create our `RegionBuilder` object.

In [ ]:
# Create region builder object #
build = RegionBuilder(sim)

Next, we will build our cytosol region mask. To do so, we will create a sphere with a radius of 200nm, centered at voxel site [32,32,32]. As with our previous tutorial, we must convert between the two using the following formula:

```math
Voxels = ActualLength / LatticeSpacing
```

In [ ]:
# Define actual radius/length (m) #
radius_actual = 2.00e-7
# Convert radius to voxels #
cyto_radius = int(np.ceil(radius_actual / lattice_spacing))
# Define boolean mask for cytosol region type using ellipsoid method #
cytosol_mask = build.ellipsoid(radius = cyto_radius, center = sim_center)

Next, we will build the inner membrane region by dialating the cytosol mask.

In [ ]:
# Dialate the cytosol mask to create an inner membrane region called the "shell" #
cyto_dilation = build.dilate(cytosol_mask, se = build.se26)
# Remove cytosol voxels from shell mask #
shell_mask = cyto_dilation & ~cytosol_mask

Next, we will create the membrane region by dialating the inner membrane region.

In [ ]:
# Dialate the shell mask to create a membrane region #
cyto_dilation = build.dilate(cyto_dilation, se = build.se26)
# Remove cytosol and shell voxels from membrane mask #
membrane_mask = cyto_dilation & ~shell_mask & ~cytosol_mask

Next, we will build the ribosome mask using a prewritten function to randomly place 500 ribosomes in cytoplasm.

In [ ]:
# Call prewritten function to randomly place 500 ribosomes in the cytosol and create a ribosome mask #
ribosomes_mask = loader.getRibosomeSites(cytosol_mask, N_edges)

Next, we will create the DNA mask by loading the DNA geometry from a precomputed file.

In [ ]:
# Define the name of the DNA file generated from b-Tree Chromo #
DNAfile = './Data/x_chain_syn3a_rep00001.bin'
# Build DNA mask with DNA geometry and prewritten function #
DNA_mask, DNA_pos = loader.getDNAsites(DNAfile, N_edges,lattice_spacing, N_2_x, N_2_y, N_2_z)
# define cyto plasm and extra cellular region
cytosol_mask = cytosol_mask & ~DNA_mask & ~ribosomes_mask
extracellular_mask = ~membrane_mask & ~cytosol_mask & ~ribosomes_mask & ~DNA_mask & ~shell_mask


Now, we will add all masks together to generate our final system architecture. We could set environment variables to the simulation region objects we want to build, or as done below, we can pass the commands to create these region objects directly to the `compose` function.

In [ ]:
# Build final system architecture and add it to the RDME simulation object #
build.compose(
    (sim.region('extracellular'), extracellular_mask),
    (sim.region('cytosol'), cytosol_mask),
    (sim.region('DNA'),DNA_mask),
    (sim.region('ribosomes'), ribosomes_mask),
    (sim.region('shell'), shell_mask),
    (sim.region('membrane'), membrane_mask))

And let's take a look at the system!

In [ ]:
# Show 2D slices of final lattice architecture #
sim.showRegionStack()

In [ ]:
# Show interactive 3D final lattice architecture #
sim.displayGeometry()

## Define Chemical Species, Initial Abundances, and Initial Positions ##
Now, we will define all chemical species, their initial abundances, and their initial locations. This process mirrors that of the previous spatially heterogeneous tutorial where we added species A, B and C.

In [ ]:
# Define all chemical species in the system #
# Define the DnaA gene #
sim.species(name = "gene", 
            texRepr = "gene", 
            annotation = "DnaA gene for RDME-GIP")
# Define the DnaA mRNA #
sim.species(name = "mRNA", 
            textRepr = "mRNA", 
            annotation = "mRNA for DnaA gene in RDME-GIP")
# Define the DnaA protein #
sim.species(name = "P", 
            texRepr = "Protein", 
            annotation = "DnaA protein in RDME-GIP")

# Place initial abundance of each chemical species in predetermined initial locations within simulation object #
# Define the location of the DnaA gene from the user-defined function to read in the DNA mask #
gene_pos = DNA_pos[0]
# Place a single copy of the DnaA gene at this predetermined location #
sim.placeNumber(sp = sim.sp.gene,
                x = gene_pos[0], 
                y = gene_pos[1], 
                z = gene_pos[2], 
                n = 1)
# Place a single copy of the RNA molecule in a random cytosol voxel #
sim.distributeNumber(sp = sim.sp.mRNA, 
                     reg = sim.reg.cytosol, 
                     count = 1)
# We will start the simulation with no protein molecules #

In [ ]:
sim.transitionRate(None, None, None, sim.diffusionZero)

## Define the Chemical Reaction Network of the System ##
Next, we will define the rate constants for the four GIP reactions we want to model: transcription, translation, RNA degradation, and protein degradation. These will be saved into the RDME simulation object using the `rateConst` function. Remember that the rates should be given in concentration-based units and jLM will convert these rates automatically depending on the reaction order. Additionally, we will use the `reaction` function in this tutorial which accomplishes the same end goal as the `region.addReaction` function. However, in the `reaction` function, we can pass a single region, multiple regions, or no regions to the `regions` flag. If no entry is given for the `regions` flag, the reaction will be available in all regions. 

In [ ]:
# Define a  reaction rate constants #
# Transcription #
sim.rateConst("trans", 6.14e-4, order=1, annotation="transcription rate")
# Translation #
sim.rateConst("transl", 7.20e-2, order=1, annotation="translation rate")
# RNA degradation #
sim.rateConst("degrad_m", 2.59e-2, order=1, annotation="mRNA degradation rate")
# Protein degradation #
sim.rateConst("degrad_p", 7.7e-6, order=1, annotation="Protein degradation rate")

# Define system chemical reaction network #
# Transcription #
sim.reaction([sim.sp.gene], [sim.sp.gene, sim.sp.mRNA], sim.rc.trans, regions=[sim.reg.DNA], annotation="transcription")
# Translation #
sim.reaction([sim.sp.mRNA], [sim.sp.mRNA, sim.sp.P], sim.rc.transl, regions=[sim.reg.ribosomes], annotation="translation")
# RNA degradation #
sim.reaction([sim.sp.mRNA], [], sim.rc.degrad_m, regions=[sim.reg.shell], annotation="mRNA degradation")
# Protein degradation #
sim.reaction([sim.sp.P], [], sim.rc.degrad_p, regions=[sim.reg.shell], annotation="Protein degradation")

## Set Diffusion Coefficients for Each Chemical Species ##
As with the previous tutorial, we will need to specify all diffusion coefficients for each species in our system. We start by setting all of them equal to zero using the `transitionRate` function and the predefined `diffusionZero` variable in our simulation object. Then, we define diffusion coefficients for each species. For simplicity, we define the diffusion coefficients for all region transitions for an individual species to be the same, however, this can be changed if one has experimental information about transitions being more or less favored between specific regions. An example of this might be that a protein diffuses slower in the membrane that the cytosol. 

A summary of the reasoning behind each species' diffusion rules is given in the [README.md](./README.md) document for this tutorial.

In [ ]:
# Set all diffusion coefficients for all species equal to zero #
sim.transitionRate(None, None, None, sim.diffusionZero)

# Define diffusion coefficients to be used for specific species #
# Define RNA diffusion coefficient #
sim.diffusionConst('mrna',4.13e-14, texRepr=r'D_{mRNA}', annotation="mRNA diffusion constant for JCVISYN3A_0001")
# Define protein diffusion coefficient #
sim.diffusionConst('protein', 0.1e-12, texRepr=r'Protein', annotation="protein diffusion co.")

# Set diffusion rules for transitions out of and into each region #
# RNA diffusion rules #
sim.transitionRate(sim.sp.mRNA, sim.reg.DNA, sim.reg.cytosol, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.DNA, sim.reg.ribosomes, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.DNA, sim.reg.shell, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.cytosol, sim.reg.cytosol, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.cytosol, sim.reg.ribosomes, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.cytosol, sim.reg.shell, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.ribosomes, sim.reg.cytosol, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.ribosomes, sim.reg.ribosomes, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.ribosomes, sim.reg.shell, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.shell, sim.reg.cytosol, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.shell, sim.reg.ribosomes, sim.dc.mrna)
sim.transitionRate(sim.sp.mRNA, sim.reg.shell, sim.reg.shell, sim.dc.mrna)
# Protein diffusion rules #
sim.transitionRate(sim.sp.P, sim.reg.cytosol, sim.reg.cytosol, sim.dc.protein)
sim.transitionRate(sim.sp.P, sim.reg.cytosol, sim.reg.shell, sim.dc.protein)
sim.transitionRate(sim.sp.P, sim.reg.ribosomes, sim.reg.cytosol, sim.dc.protein)
sim.transitionRate(sim.sp.P, sim.reg.ribosomes, sim.reg.ribosomes, sim.dc.protein)
sim.transitionRate(sim.sp.P, sim.reg.ribosomes, sim.reg.shell, sim.dc.protein)
sim.transitionRate(sim.sp.P, sim.reg.shell, sim.reg.cytosol, sim.dc.protein)
sim.transitionRate(sim.sp.P, sim.reg.shell, sim.reg.shell, sim.dc.protein)

Finally, now that we have all chemical species, initial abundances, chemical reactions, reaction rate constants, and diffusion coefficients defined, we can check our system and simulation specifications before performing the simulation.

In [ ]:
# Check all species, abundances, chemical reactions, and diffusion coefficients #
sim.showAllSpecies()

In [ ]:
# Check all system and simulation parameters #
sim

## Run the RDME Simulation ##
Now, we will run our `finalize` function to tell the simulation object that we are ready to perform a simluation and then run the simulation with the `sim.run` function.

In [ ]:
# Finalize the simulation state #
sim.finalize()
# Run the RDME simulation #
sim.run(solver=IntMpdRdmeSolver(), cudaDevices=[0])

An important note about the RDME solver in Lattice Microbes is that, by default, it performs a single simulation replicate, which is stored as the first replicate in the output file. If additional replicates are desired, one must specify the replicate that is being run so that previous simulations are not overwritten.

## Analysis of Simulation Data ##
Congratulations, you have run your first GIP-RDME simulation in LM!

The output of this simulation is saved as a `.lm` file, which is in h5 format. We can use the `RDME.File` function which we have imported as `RDMEFile` to import our species trajectories into our python session. In this function, we will use the `getNumberTrajectory` method as we did in our previous tutorial. The method returns two `numpy` arrays: `ts` (evaluation times in seconds, shape `(nt,)`) and `counts` (particle counts, shape `(nt,)`).

In [ ]:
# Read in RDME simulation output file as object named traj #
traj = RDMEFile(fname = "./Results/TutR2.GIP_result_6300s.lm", replicate=1)
# Define timepoint array and species abundance arrays #
ts, genes = traj.getNumberTrajectory(species="gene")
ts, mRNAs = traj.getNumberTrajectory(species="mRNA")
ts, proteins = traj.getNumberTrajectory(species="P")

# Visualize the species trajectories #
sns.set(style="ticks")
palette = sns.color_palette("husl", 3)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(ts, mRNAs,  label='mRNA',    color=palette[1])
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('mRNA counts', color='black')
ax1.tick_params(axis='y')
ax2 = ax1.twinx()
ax2.plot(ts, proteins, label='protein', color=palette[2])
ax2.set_ylabel('Protein counts', color=palette[2])
ax2.tick_params(axis='y', labelcolor=palette[2])
ax1.grid(False)
ax2.grid(False)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.title('Trajectories of Genetic Information Processing')
plt.tight_layout()
plt.savefig('./Plots/TutR2_GIP.png')
plt.show()